# Haryanvi Translation Notebook

This notebook fine-tunes one Gemma 4 adapter for Hindi-to-Haryanvi translation.

In [ ]:
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional
import os

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


@dataclass
class CombinedConfig:
    # -------- Authentication --------
    HF_TOKEN: Optional[str] = os.getenv("HF_TOKEN")

    # -------- Shared --------
    RANDOM_SEED: int = 42
    PROJECT_DIR: Path = Path(".")
    USE_GOOGLE_DRIVE: bool = True
    GOOGLE_DRIVE_ROOT: Path = Path("/content/drive/MyDrive/DSAI_translation")
    USE_LOCAL_RUNTIME_HF_CACHE: bool = True
    COLAB_LOCAL_CACHE_ROOT: Path = Path("/content/hf_cache")
    HF_HOME_DIR: Path = Path("combined_outputs/hf_home")
    HF_HUB_CACHE_DIR: Path = Path("combined_outputs/hf_hub_cache")
    TRANSFORMERS_CACHE_DIR: Path = Path("combined_outputs/transformers_cache")

    # -------- Text / LLM data --------
    HF_TEXT_REPO_ID: str = "Satyam-Srivastava/RDS"
    HF_TEXT_REPO_TYPE: str = "dataset"
    TXT_HINDI_FILE: str = "text_data/hindi1.txt"
    TXT_HARYANVI_FILE: str = "text_data/haryanvi1.txt"
    JSON_FILE: str = "text_data/hindi_haryanvi_dataset_5000.json"
    TEXT_MIN_WORDS: int = 3
    TEXT_MAX_WORDS: int = 25
    TEXT_TEST_SPLIT: float = 0.20
    TEXT_VAL_SPLIT: float = 0.50

    # -------- LLM fine-tuning --------
    LLM_MODEL_ID: str = "google/gemma-4-E4B-it"
    LLM_MAX_LENGTH: int = 256
    LLM_AUTO_MAX_LENGTH_FROM_DATA: bool = True
    LLM_MAX_LENGTH_QUANTUM: int = 64
    LLM_MAX_LENGTH_CAP: int = 512
    LLM_OUTPUT_ROOT: Path = Path("combined_outputs/gemma4_e4b_llm")
    LLM_ENABLE_THINKING: bool = False
    LLM_GENERATION_MAX_NEW_TOKENS: int = 64
    LLM_TEST_LIMIT: Optional[int] = None
    LLM_QUALITATIVE_HOLDOUT_SIZE: int = 25
    LLM_RUN_FINAL_TEST_EVAL: bool = True
    LLM_SKIP_FINISHED_RUNS: bool = True
    LLM_FINE_TUNE_RUNS: List[Dict[str, Any]] = field(
        default_factory=lambda: [
            {
                "name": "gemma4_haryanvi_a100_qlora",
                "learning_rate": 1e-4,
                "num_train_epochs": 4,
                "per_device_train_batch_size": 4,
                "gradient_accumulation_steps": 4,
                "per_device_eval_batch_size": 4,
                "lora_r": 32,
                "lora_alpha": 64,
                "lora_dropout": 0.05,
                "warmup_steps": 100,
                "weight_decay": 0.01,
                "max_grad_norm": 0.30,
            },
        ]
    )

    # -------- Execution toggles --------
    RUN_LLM_FINE_TUNE: bool = True
    RUN_LLM_INTERACTIVE_TEST: bool = True


CFG = CombinedConfig()

def configure_project_paths(base_dir: Path, cache_root: Optional[Path] = None) -> None:
    base_dir = Path(base_dir)
    cache_root = Path(cache_root) if cache_root is not None else (base_dir / "combined_outputs")
    CFG.PROJECT_DIR = base_dir
    CFG.LLM_OUTPUT_ROOT = base_dir / "combined_outputs" / "gemma4_e4b_llm"
    CFG.HF_HOME_DIR = cache_root / "hf_home"
    CFG.HF_HUB_CACHE_DIR = cache_root / "hf_hub_cache"
    CFG.TRANSFORMERS_CACHE_DIR = cache_root / "transformers_cache"

    for path_attr in [
        "PROJECT_DIR",
        "LLM_OUTPUT_ROOT",
        "HF_HOME_DIR",
        "HF_HUB_CACHE_DIR",
        "TRANSFORMERS_CACHE_DIR",
    ]:
        path_value = getattr(CFG, path_attr)
        if isinstance(path_value, Path):
            path_value.mkdir(parents=True, exist_ok=True)

    # Keep model/download caches configurable so Colab can use fast local disk.
    os.environ["HF_HOME"] = str(CFG.HF_HOME_DIR)
    os.environ["HF_HUB_CACHE"] = str(CFG.HF_HUB_CACHE_DIR)
    os.environ["TRANSFORMERS_CACHE"] = str(CFG.TRANSFORMERS_CACHE_DIR)


try:
    from google.colab import drive as google_drive

    IN_COLAB = True
except Exception:
    google_drive = None
    IN_COLAB = False

if IN_COLAB and CFG.USE_GOOGLE_DRIVE:
    # Mount once so checkpoints and downloaded assets survive runtime disconnects.
    google_drive.mount("/content/drive", force_remount=False)
    cache_root = CFG.COLAB_LOCAL_CACHE_ROOT if CFG.USE_LOCAL_RUNTIME_HF_CACHE else (CFG.GOOGLE_DRIVE_ROOT / "combined_outputs")
    configure_project_paths(CFG.GOOGLE_DRIVE_ROOT, cache_root=cache_root)
    print(f"Google Drive mounted. Saving checkpoints and outputs to: {CFG.PROJECT_DIR}")
    print(f"Hugging Face caches stored at: {CFG.HF_HOME_DIR.parent}")
else:
    configure_project_paths(CFG.PROJECT_DIR.resolve())
    print(f"Saving checkpoints and outputs to local path: {CFG.PROJECT_DIR}")

print("Top-level config loaded.")
print("Adjust CFG before running expensive cells if needed.")
print(asdict(CFG))

In [ ]:
# Run this once per fresh environment.
!pip install -q --upgrade pip setuptools wheel
!pip install -q -U transformers accelerate huggingface_hub trl peft bitsandbytes datasets pandas scikit-learn sacrebleu matplotlib


In [ ]:
import copy
import gc
import json
import platform
import random
import re
import shutil
import sys
import warnings

import datasets as datasets_lib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import peft
import sacrebleu
import torch
import transformers
import trl
from datasets import Dataset, DatasetDict, load_dataset
from huggingface_hub import hf_hub_download
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from sacrebleu.metrics import BLEU, CHRF
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForCausalLM,
    AutoProcessor,
    BitsAndBytesConfig,
    default_data_collator,
)
from trl import SFTConfig, SFTTrainer

warnings.filterwarnings("ignore")


def resolve_hf_token(seed_value: Optional[str]) -> Optional[str]:
    if seed_value:
        return seed_value
    env_token = os.getenv("HF_TOKEN")
    if env_token:
        return env_token
    try:
        from google.colab import userdata

        colab_token = userdata.get("HF_TOKEN")
        if colab_token:
            return colab_token
    except Exception:
        pass
    return None


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


CFG.HF_TOKEN = resolve_hf_token(CFG.HF_TOKEN)
if not CFG.HF_TOKEN:
    raise ValueError("Set HF_TOKEN in the environment or Colab userdata before downloading gated assets.")

set_seed(CFG.RANDOM_SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
IS_BF16_CAPABLE = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
IS_A100 = "A100" in GPU_NAME.upper()

print(f"Device: {GPU_NAME}")
print(f"BF16 capable: {IS_BF16_CAPABLE}")
print(f"A100 detected: {IS_A100}")

## Gemma 4 Fine-Tuning

The next cells:
- download and clean the text dataset,
- count tokens with the Gemma 4 tokenizer and choose the training context length,
- tokenize it for Gemma 4 chat-format supervised fine-tuning,
- train one A100-oriented QLoRA adapter,
- save validation predictions and metrics,
- keep one fixed qualitative review table,
- evaluate the final adapter on the test split.

In [ ]:
print("--- Loading and cleaning Hindi/Haryanvi text data ---")

hindi_txt_path = hf_hub_download(
    repo_id=CFG.HF_TEXT_REPO_ID,
    filename=CFG.TXT_HINDI_FILE,
    repo_type=CFG.HF_TEXT_REPO_TYPE,
    token=CFG.HF_TOKEN,
)
haryanvi_txt_path = hf_hub_download(
    repo_id=CFG.HF_TEXT_REPO_ID,
    filename=CFG.TXT_HARYANVI_FILE,
    repo_type=CFG.HF_TEXT_REPO_TYPE,
    token=CFG.HF_TOKEN,
)
json_path = hf_hub_download(
    repo_id=CFG.HF_TEXT_REPO_ID,
    filename=CFG.JSON_FILE,
    repo_type=CFG.HF_TEXT_REPO_TYPE,
    token=CFG.HF_TOKEN,
)

with open(hindi_txt_path, "r", encoding="utf-8") as f:
    hindi_lines = [line.strip() for line in f if line.strip()]
with open(haryanvi_txt_path, "r", encoding="utf-8") as f:
    haryanvi_lines = [line.strip() for line in f if line.strip()]

min_len = min(len(hindi_lines), len(haryanvi_lines))
df_txt = pd.DataFrame(
    {
        "hindi": hindi_lines[:min_len],
        "haryanvi": haryanvi_lines[:min_len],
    }
)

with open(json_path, "r", encoding="utf-8") as f:
    json_data = json.load(f)

df_json = pd.DataFrame(json_data).rename(columns={"Hindi": "hindi", "Haryanvi": "haryanvi"})
df_text = pd.concat([df_txt, df_json], ignore_index=True)
df_text = df_text.dropna(subset=["hindi", "haryanvi"]).drop_duplicates().reset_index(drop=True)

df_text["hindi"] = df_text["hindi"].astype(str).str.strip()
df_text["haryanvi"] = df_text["haryanvi"].astype(str).str.strip()
df_text["hindi_len"] = df_text["hindi"].str.split().str.len()
df_text = df_text[
    (df_text["hindi_len"] >= CFG.TEXT_MIN_WORDS)
    & (df_text["hindi_len"] <= CFG.TEXT_MAX_WORDS)
].reset_index(drop=True)

train_df, temp_df = train_test_split(
    df_text,
    test_size=CFG.TEXT_TEST_SPLIT,
    random_state=CFG.RANDOM_SEED,
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=CFG.TEXT_VAL_SPLIT,
    random_state=CFG.RANDOM_SEED,
)

qualitative_holdout_size = min(CFG.LLM_QUALITATIVE_HOLDOUT_SIZE, max(0, len(val_df) - 1))
if qualitative_holdout_size > 0:
    qualitative_holdout_df = val_df.sample(
        n=qualitative_holdout_size,
        random_state=CFG.RANDOM_SEED,
    ).copy()
    validation_df = val_df.drop(index=qualitative_holdout_df.index).copy()
else:
    qualitative_holdout_df = val_df.iloc[0:0].copy()
    validation_df = val_df.copy()

for frame in [train_df, validation_df, qualitative_holdout_df, test_df]:
    frame.reset_index(drop=True, inplace=True)

text_dataset_dict = DatasetDict(
    {
        "train": Dataset.from_pandas(train_df, preserve_index=False),
        "validation": Dataset.from_pandas(validation_df, preserve_index=False),
        "qualitative_holdout": Dataset.from_pandas(qualitative_holdout_df, preserve_index=False),
        "test": Dataset.from_pandas(test_df, preserve_index=False),
    }
)

print(
    f"Train: {len(train_df)} | Validation: {len(validation_df)} | "
    f"Qualitative Holdout: {len(qualitative_holdout_df)} | Test: {len(test_df)}"
)
qualitative_holdout_path = CFG.LLM_OUTPUT_ROOT / "qualitative_holdout.csv"
qualitative_holdout_df.to_csv(qualitative_holdout_path, index=False)
print(f"Saved fixed qualitative holdout: {qualitative_holdout_path}")
display(qualitative_holdout_df.head() if len(qualitative_holdout_df) else validation_df.head())

In [ ]:
processor = AutoProcessor.from_pretrained(CFG.LLM_MODEL_ID, token=CFG.HF_TOKEN)
tokenizer = getattr(processor, "tokenizer", processor)
if getattr(tokenizer, "pad_token", None) is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


def apply_gemma_chat_template(messages: List[Dict[str, str]], add_generation_prompt: bool = False) -> str:
    template_kwargs = {
        "tokenize": False,
        "add_generation_prompt": add_generation_prompt,
    }
    try:
        return processor.apply_chat_template(
            messages,
            enable_thinking=CFG.LLM_ENABLE_THINKING,
            **template_kwargs,
        )
    except TypeError:
        return processor.apply_chat_template(messages, **template_kwargs)


def encode_text(text: str, **kwargs) -> Dict[str, Any]:
    return processor(text=text, **kwargs)


def build_chat_prompt(hindi_text: str, haryanvi_text: Optional[str] = None) -> str:
    dialog = [
        {
            "role": "system",
            "content": (
                "You are a precise Hindi to Haryanvi (Bangru) translator. "
                "Convert the standard Hindi input into grammatically correct Haryanvi. "
                "Return only the translated Haryanvi sentence."
            ),
        },
        {"role": "user", "content": hindi_text},
    ]
    if haryanvi_text is not None:
        dialog.append({"role": "assistant", "content": haryanvi_text})
        return apply_gemma_chat_template(dialog, add_generation_prompt=False)
    return apply_gemma_chat_template(dialog, add_generation_prompt=True)


def tokenize_for_training(batch: Dict[str, List[str]], max_length: int) -> Dict[str, Any]:
    input_ids_list = []
    attention_masks_list = []
    labels_list = []

    for hindi, haryanvi in zip(batch["hindi"], batch["haryanvi"]):
        prompt_only = build_chat_prompt(hindi)
        full_prompt = build_chat_prompt(hindi, haryanvi)

        prompt_tokens = encode_text(
            prompt_only,
            max_length=max_length,
            padding="max_length",
            truncation=True,
        )
        full_tokens = encode_text(
            full_prompt,
            max_length=max_length,
            padding="max_length",
            truncation=True,
        )

        input_ids = full_tokens["input_ids"]
        attention_mask = full_tokens["attention_mask"]
        labels = input_ids.copy()
        prompt_length = int(sum(prompt_tokens["attention_mask"]))

        # Only supervise the assistant response; the prompt stays as context.
        for idx in range(min(prompt_length, len(labels))):
            labels[idx] = -100

        for idx, mask in enumerate(attention_mask):
            if mask == 0:
                labels[idx] = -100

        input_ids_list.append(input_ids)
        attention_masks_list.append(attention_mask)
        labels_list.append(labels)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_masks_list,
        "labels": labels_list,
    }


TOKENIZED_DATASET_CACHE: Dict[int, DatasetDict] = {}


def get_tokenized_dataset(max_length: int) -> DatasetDict:
    normalized_max_length = int(max_length)
    if normalized_max_length not in TOKENIZED_DATASET_CACHE:
        # Cache tokenized splits so context-length experiments do not redo the same work.
        TOKENIZED_DATASET_CACHE[normalized_max_length] = text_dataset_dict.map(
            lambda batch: tokenize_for_training(batch, max_length=normalized_max_length),
            batched=True,
            remove_columns=list(text_dataset_dict["train"].column_names),
        )
    return TOKENIZED_DATASET_CACHE[normalized_max_length]



In [ ]:
def estimate_token_length(text: str) -> int:
    return len(
        encode_text(
            text,
            truncation=False,
        )["input_ids"]
    )


def round_up_to_multiple(value: int, multiple: int) -> int:
    return int(((value + multiple - 1) // multiple) * multiple)


truncation_analysis_df = pd.DataFrame(
    {
        "split": ["train"] * len(train_df) + ["validation"] * len(validation_df) + ["test"] * len(test_df),
        "full_prompt_tokens": [
            estimate_token_length(build_chat_prompt(row.hindi, row.haryanvi))
            for row in train_df.itertuples(index=False)
        ]
        + [
            estimate_token_length(build_chat_prompt(row.hindi, row.haryanvi))
            for row in validation_df.itertuples(index=False)
        ]
        + [
            estimate_token_length(build_chat_prompt(row.hindi, row.haryanvi))
            for row in test_df.itertuples(index=False)
        ],
    }
)

observed_max_tokens = int(truncation_analysis_df["full_prompt_tokens"].max())
if CFG.LLM_AUTO_MAX_LENGTH_FROM_DATA:
    data_driven_max_length = round_up_to_multiple(observed_max_tokens, CFG.LLM_MAX_LENGTH_QUANTUM)
    CFG.LLM_MAX_LENGTH = min(
        max(CFG.LLM_MAX_LENGTH, data_driven_max_length),
        CFG.LLM_MAX_LENGTH_CAP,
    )

truncation_analysis_df["is_truncated"] = truncation_analysis_df["full_prompt_tokens"] > CFG.LLM_MAX_LENGTH
truncation_summary_df = (
    truncation_analysis_df.groupby("split", as_index=False)
    .agg(
        samples=("full_prompt_tokens", "size"),
        avg_tokens=("full_prompt_tokens", "mean"),
        p95_tokens=("full_prompt_tokens", lambda values: float(np.percentile(values, 95))),
        max_tokens=("full_prompt_tokens", "max"),
        truncated_samples=("is_truncated", "sum"),
    )
)
truncation_summary_df["truncated_pct"] = (
    100.0 * truncation_summary_df["truncated_samples"] / truncation_summary_df["samples"]
)
print(f"Gemma 4 tokenizer max observed input+target tokens: {observed_max_tokens}")
print(f"Using CFG.LLM_MAX_LENGTH = {CFG.LLM_MAX_LENGTH}")
display(truncation_summary_df)

llm_ready_dataset = get_tokenized_dataset(CFG.LLM_MAX_LENGTH)
print("Tokenized datasets ready for Gemma 4 training.")

sample_batch = llm_ready_dataset["train"][0]
supervised_token_ids = [
    token_id for token_id, label in zip(sample_batch["input_ids"], sample_batch["labels"]) if label != -100
]
ignored_positions = sum(label == -100 for label in sample_batch["labels"])
print(f"Assistant-only label check: ignored positions = {ignored_positions}, supervised tokens = {len(supervised_token_ids)}")
print("Decoded supervised assistant span:")
print(tokenizer.decode(supervised_token_ids, skip_special_tokens=False))


In [ ]:
bleu_metric = BLEU(effective_order=True)
chrf_metric = CHRF(word_order=2)


def normalize_for_compare(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def compute_translation_metrics(references: List[str], predictions: List[str]) -> Dict[str, float]:
    normalized_references = [normalize_for_compare(text) for text in references]
    normalized_predictions = [normalize_for_compare(text) for text in predictions]
    if not normalized_predictions:
        return {
            "exact_match_rate": float("nan"),
            "corpus_bleu": float("nan"),
            "corpus_chrf": float("nan"),
        }

    exact_match_rate = float(
        sum(pred == ref for pred, ref in zip(normalized_predictions, normalized_references))
        / len(normalized_predictions)
    )
    corpus_bleu = float(bleu_metric.corpus_score(normalized_predictions, [normalized_references]).score)
    corpus_chrf = float(chrf_metric.corpus_score(normalized_predictions, [normalized_references]).score)
    return {
        "exact_match_rate": exact_match_rate,
        "corpus_bleu": corpus_bleu,
        "corpus_chrf": corpus_chrf,
    }


def collect_package_versions() -> Dict[str, str]:
    return {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "trl": trl.__version__,
        "peft": peft.__version__,
        "datasets": datasets_lib.__version__,
        "sacrebleu": sacrebleu.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
    }


def build_run_manifest(experiment_name: str, experiment_cfg: Dict[str, Any], experiment_max_length: int) -> Dict[str, Any]:
    return {
        "experiment_name": experiment_name,
        "seed": CFG.RANDOM_SEED,
        "model_id": CFG.LLM_MODEL_ID,
        "llm_max_length": int(experiment_max_length),
        "gpu_name": GPU_NAME,
        "is_bf16_capable": bool(IS_BF16_CAPABLE),
        "is_a100": bool(IS_A100),
        "package_versions": collect_package_versions(),
        "fine_tune_config": dict(experiment_cfg),
        "python_executable": sys.executable,
    }


def build_quant_config() -> BitsAndBytesConfig:
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if IS_BF16_CAPABLE else torch.float16,
    )


def load_base_llm():
    model = AutoModelForCausalLM.from_pretrained(
        CFG.LLM_MODEL_ID,
        quantization_config=build_quant_config(),
        device_map="auto",
        token=CFG.HF_TOKEN,
        attn_implementation="sdpa",
        dtype="auto",
    )
    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model)
    return model


def attach_lora_adapter(model, experiment_cfg: Dict[str, Any]):
    peft_config = LoraConfig(
        r=experiment_cfg["lora_r"],
        lora_alpha=experiment_cfg["lora_alpha"],
        lora_dropout=experiment_cfg["lora_dropout"],
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()
    return model


def parse_gemma_response(inference_processor, response_text: str) -> str:
    parse_response = getattr(inference_processor, "parse_response", None)
    if callable(parse_response):
        try:
            parsed = parse_response(response_text)
            if isinstance(parsed, str):
                response_text = parsed
            elif isinstance(parsed, dict):
                response_text = (
                    parsed.get("answer")
                    or parsed.get("response")
                    or parsed.get("content")
                    or response_text
                )
        except Exception:
            pass

    response_text = re.sub(r"<\|channel\>thought.*?<channel\|>", "", response_text, flags=re.DOTALL)
    response_text = re.sub(r"<\|[^>]+\|>", "", response_text)
    return response_text.strip()


def generate_translation(model, inference_processor, hindi_text: str) -> str:
    prompt = build_chat_prompt(hindi_text)
    inputs = inference_processor(text=prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=CFG.LLM_GENERATION_MAX_NEW_TOKENS,
            pad_token_id=tokenizer.pad_token_id,
            do_sample=False,
            repetition_penalty=1.15,
        )

    response_text = inference_processor.decode(outputs[0][input_len:], skip_special_tokens=False)
    prediction = parse_gemma_response(inference_processor, response_text)
    prediction = prediction.split("\n")[0].split("|")[0].strip()
    if "।" in prediction:
        prediction = prediction.split("।")[0].strip() + "।"
    return prediction


def run_predictions_on_dataframe(
    model,
    inference_processor,
    dataset_df: pd.DataFrame,
    experiment_name: str,
    split_name: str,
) -> pd.DataFrame:
    prediction_rows = []
    working_df = dataset_df.copy()
    if split_name == "test" and CFG.LLM_TEST_LIMIT:
        working_df = working_df.head(CFG.LLM_TEST_LIMIT).copy()

    for sample_id, row in working_df.reset_index(drop=True).iterrows():
        prediction = generate_translation(model, inference_processor, row["hindi"])
        reference = normalize_for_compare(row["haryanvi"])
        normalized_prediction = normalize_for_compare(prediction)
        prediction_rows.append(
            {
                "sample_id": sample_id,
                "split_name": split_name,
                "hindi": row["hindi"],
                "reference_haryanvi": row["haryanvi"],
                "prediction": prediction,
                "exact_match": normalized_prediction == reference,
                "sentence_chrf": float(chrf_metric.sentence_score(normalized_prediction, [reference]).score),
                "experiment_name": experiment_name,
            }
        )

    return pd.DataFrame(prediction_rows)


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def run_single_llm_experiment(experiment_cfg: Dict[str, Any]) -> Dict[str, Any]:
    experiment_name = experiment_cfg["name"]
    experiment_max_length = int(experiment_cfg.get("llm_max_length", CFG.LLM_MAX_LENGTH))
    experiment_dataset = get_tokenized_dataset(experiment_max_length)
    output_dir = CFG.LLM_OUTPUT_ROOT / experiment_name
    adapter_dir = output_dir / "adapter"
    validation_predictions_path = output_dir / "validation_predictions.csv"
    qualitative_predictions_path = output_dir / "qualitative_predictions.csv"
    log_history_path = output_dir / "training_log_history.csv"
    worst_cases_path = output_dir / "worst_validation_cases.csv"
    manifest_path = output_dir / "run_manifest.json"
    summary_path = output_dir / "summary.json"
    output_dir.mkdir(parents=True, exist_ok=True)

    # Save both the fine-tune config and a fuller run manifest for reproducibility.
    with open(output_dir / "fine_tune_config.json", "w", encoding="utf-8") as f:
        json.dump(experiment_cfg, f, ensure_ascii=False, indent=2)
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(build_run_manifest(experiment_name, experiment_cfg, experiment_max_length), f, ensure_ascii=False, indent=2)

    if CFG.LLM_SKIP_FINISHED_RUNS and validation_predictions_path.exists() and qualitative_predictions_path.exists() and summary_path.exists():
        print(f"Skipping {experiment_name} because saved outputs already exist.")
        with open(summary_path, "r", encoding="utf-8") as f:
            summary = json.load(f)
        validation_predictions_df = pd.read_csv(validation_predictions_path)
        if "sentence_chrf" in validation_predictions_df.columns and not worst_cases_path.exists():
            validation_predictions_df.nsmallest(20, "sentence_chrf").to_csv(worst_cases_path, index=False)
        summary.setdefault("llm_max_length", experiment_max_length)
        summary["run_manifest_path"] = str(manifest_path)
        summary["worst_validation_cases_path"] = str(worst_cases_path)
        with open(summary_path, "w", encoding="utf-8") as f:
            json.dump(summary, f, ensure_ascii=False, indent=2)
        summary["validation_predictions_df"] = validation_predictions_df
        summary["qualitative_predictions_df"] = pd.read_csv(qualitative_predictions_path)
        return summary

    model = load_base_llm()
    model = attach_lora_adapter(model, experiment_cfg)

    training_args = SFTConfig(
        output_dir=str(output_dir / "checkpoints"),
        per_device_train_batch_size=experiment_cfg["per_device_train_batch_size"],
        gradient_accumulation_steps=experiment_cfg["gradient_accumulation_steps"],
        per_device_eval_batch_size=experiment_cfg["per_device_eval_batch_size"],
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        logging_steps=10,
        learning_rate=experiment_cfg["learning_rate"],
        warmup_steps=experiment_cfg["warmup_steps"],
        num_train_epochs=experiment_cfg["num_train_epochs"],
        lr_scheduler_type="cosine",
        max_grad_norm=experiment_cfg.get("max_grad_norm", 0.3),
        weight_decay=experiment_cfg.get("weight_decay", 0.0),
        optim="paged_adamw_32bit",
        bf16=IS_BF16_CAPABLE,
        fp16=torch.cuda.is_available() and not IS_BF16_CAPABLE,
        report_to="none",
        remove_unused_columns=False,
        group_by_length=True,
        seed=CFG.RANDOM_SEED,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        max_length=experiment_max_length,
        dataset_text_field=None,
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=experiment_dataset["train"],
        eval_dataset=experiment_dataset["validation"],
        processing_class=tokenizer,
        data_collator=default_data_collator,
    )

    train_result = trainer.train()
    eval_metrics = trainer.evaluate()
    pd.DataFrame(trainer.state.log_history).to_csv(log_history_path, index=False)

    trainer.model.save_pretrained(adapter_dir)
    processor.save_pretrained(adapter_dir)

    validation_predictions_df = run_predictions_on_dataframe(
        trainer.model,
        processor,
        validation_df,
        experiment_name,
        "validation",
    )
    qualitative_predictions_df = run_predictions_on_dataframe(
        trainer.model,
        processor,
        qualitative_holdout_df,
        experiment_name,
        "qualitative_holdout",
    )
    validation_predictions_df.to_csv(validation_predictions_path, index=False)
    qualitative_predictions_df.to_csv(qualitative_predictions_path, index=False)
    # Keep a small failure-analysis table for quick review after each run.
    validation_predictions_df.nsmallest(20, "sentence_chrf").to_csv(worst_cases_path, index=False)

    validation_metrics = compute_translation_metrics(
        validation_predictions_df["reference_haryanvi"].tolist(),
        validation_predictions_df["prediction"].tolist(),
    )
    qualitative_metrics = compute_translation_metrics(
        qualitative_predictions_df["reference_haryanvi"].tolist(),
        qualitative_predictions_df["prediction"].tolist(),
    )

    summary = {
        "experiment_name": experiment_name,
        "eval_loss": float(eval_metrics.get("eval_loss", float("nan"))),
        "train_runtime": float(train_result.metrics.get("train_runtime", float("nan"))),
        "train_samples_per_second": float(train_result.metrics.get("train_samples_per_second", float("nan"))),
        "validation_exact_match_rate": float(validation_metrics["exact_match_rate"]),
        "validation_corpus_bleu": float(validation_metrics["corpus_bleu"]),
        "validation_corpus_chrf": float(validation_metrics["corpus_chrf"]),
        "qualitative_exact_match_rate": float(qualitative_metrics["exact_match_rate"]),
        "qualitative_corpus_bleu": float(qualitative_metrics["corpus_bleu"]),
        "qualitative_corpus_chrf": float(qualitative_metrics["corpus_chrf"]),
        "adapter_dir": str(adapter_dir),
        "llm_max_length": experiment_max_length,
        "weight_decay": float(experiment_cfg.get("weight_decay", 0.0)),
        "max_grad_norm": float(experiment_cfg.get("max_grad_norm", 0.3)),
        "lora_dropout": float(experiment_cfg.get("lora_dropout", 0.0)),
        "training_log_history_path": str(log_history_path),
        "run_manifest_path": str(manifest_path),
        "worst_validation_cases_path": str(worst_cases_path),
    }
    summary.update(experiment_cfg)

    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    summary["validation_predictions_df"] = validation_predictions_df
    summary["qualitative_predictions_df"] = qualitative_predictions_df

    del trainer
    del model
    cleanup_cuda()
    return summary


def load_adapter_for_inference(adapter_dir: Path):
    base_model = AutoModelForCausalLM.from_pretrained(
        CFG.LLM_MODEL_ID,
        quantization_config=build_quant_config(),
        device_map="auto",
        token=CFG.HF_TOKEN,
        attn_implementation="sdpa",
        dtype="auto",
    )
    merged_model = PeftModel.from_pretrained(base_model, str(adapter_dir))
    merged_model.eval()
    return merged_model

In [ ]:
if not CFG.RUN_LLM_FINE_TUNE:
    print("Skipping LLM fine-tuning because CFG.RUN_LLM_FINE_TUNE is False.")
else:
    fine_tune_cfg = CFG.LLM_FINE_TUNE_RUNS[0]
    print(f"\n=== Fine-tuning run: {fine_tune_cfg['name']} ===")
    summary = run_single_llm_experiment(fine_tune_cfg)
    qualitative_predictions_df = summary.pop("qualitative_predictions_df")
    summary.pop("validation_predictions_df", None)

    llm_summary_df = pd.DataFrame([summary])
    fine_tune_summary_path = CFG.LLM_OUTPUT_ROOT / "fine_tune_summary.csv"
    qualitative_review_path = CFG.LLM_OUTPUT_ROOT / "qualitative_review.csv"

    qualitative_review_df = qualitative_holdout_df.copy().reset_index(drop=True)
    qualitative_review_df.insert(0, "sample_id", qualitative_review_df.index)
    qualitative_review_df.rename(columns={"haryanvi": "reference_haryanvi"}, inplace=True)
    qualitative_review_df = qualitative_review_df.merge(
        qualitative_predictions_df[["sample_id", "prediction"]].rename(
            columns={"prediction": "fine_tuned_prediction"}
        ),
        on="sample_id",
        how="left",
    )

    llm_summary_df.to_csv(fine_tune_summary_path, index=False)
    qualitative_review_df.to_csv(qualitative_review_path, index=False)

    print(f"Saved fine-tune summary: {fine_tune_summary_path}")
    print(f"Saved qualitative review table: {qualitative_review_path}")
    display(llm_summary_df)
    display(qualitative_review_df.head(20))

    worst_cases_path = Path(llm_summary_df.iloc[0]["worst_validation_cases_path"])
    if worst_cases_path.exists():
        print(f"Worst validation examples: {worst_cases_path}")
        display(pd.read_csv(worst_cases_path).head(20))


In [ ]:
if CFG.RUN_LLM_FINE_TUNE:
    metric_columns = [
        "experiment_name",
        "eval_loss",
        "validation_exact_match_rate",
        "validation_corpus_bleu",
        "validation_corpus_chrf",
    ]
    available_metric_columns = [column for column in metric_columns if column in llm_summary_df.columns]
    display(llm_summary_df[available_metric_columns])

    metric_plot_df = pd.DataFrame(
        {
            "metric": ["chrF", "BLEU", "Exact match %"],
            "value": [
                float(llm_summary_df.iloc[0]["validation_corpus_chrf"]),
                float(llm_summary_df.iloc[0]["validation_corpus_bleu"]),
                100.0 * float(llm_summary_df.iloc[0]["validation_exact_match_rate"]),
            ],
        }
    )
    ax = metric_plot_df.plot.bar(x="metric", y="value", legend=False, figsize=(8, 4))
    ax.set_title("Validation Metrics")
    ax.set_ylabel("Score")
    ax.tick_params(axis="x", rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("Skipping Gemma 4 metric plot because fine-tuning was not run.")


In [ ]:
if CFG.RUN_LLM_INTERACTIVE_TEST and CFG.RUN_LLM_FINE_TUNE:
    fine_tune_run_name = llm_summary_df.iloc[0]["experiment_name"]
    adapter_dir = Path(llm_summary_df.iloc[0]["adapter_dir"])
    print(f"Fine-tuned adapter: {fine_tune_run_name}")

    finetuned_inference_model = load_adapter_for_inference(adapter_dir)

    demo_sentences = [
        "मैं आज बाजार जा रहा हूँ।",
        "क्या तुम कल स्कूल जाओगे?",
        "यह किताब मेरी है, इसे मत छूना।",
    ]

    for sentence in demo_sentences:
        print(f"Standard Hindi: {sentence}")
        print(f"Fine-Tuned Model Output: {generate_translation(finetuned_inference_model, processor, sentence)}\n")

    best_adapter_archive = CFG.LLM_OUTPUT_ROOT / f"{fine_tune_run_name}_adapter"
    shutil.make_archive(str(best_adapter_archive), "zip", adapter_dir)
    try:
        from google.colab import files
        files.download(f"{best_adapter_archive}.zip")
    except Exception:
        print(f"Created adapter archive: {best_adapter_archive}.zip")

    if CFG.LLM_RUN_FINAL_TEST_EVAL:
        final_test_predictions_df = run_predictions_on_dataframe(
            finetuned_inference_model,
            processor,
            test_df,
            fine_tune_run_name,
            "test",
        )
        final_test_predictions_path = CFG.LLM_OUTPUT_ROOT / "final_test_predictions.csv"
        final_test_summary_path = CFG.LLM_OUTPUT_ROOT / "final_test_summary.json"
        final_test_predictions_df.to_csv(final_test_predictions_path, index=False)
        final_test_metrics = compute_translation_metrics(
            final_test_predictions_df["reference_haryanvi"].tolist(),
            final_test_predictions_df["prediction"].tolist(),
        )
        with open(final_test_summary_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "fine_tune_run_name": fine_tune_run_name,
                    "test_exact_match_rate": float(final_test_metrics["exact_match_rate"]),
                    "test_corpus_bleu": float(final_test_metrics["corpus_bleu"]),
                    "test_corpus_chrf": float(final_test_metrics["corpus_chrf"]),
                },
                f,
                ensure_ascii=False,
                indent=2,
            )
        print(f"Saved final test predictions: {final_test_predictions_path}")
        print(f"Saved final test summary: {final_test_summary_path}")

    del finetuned_inference_model
    cleanup_cuda()
else:
    print("Interactive LLM test skipped.")


## Optional: export merged HF model and Q4_K_S GGUF

Run these cells after fine-tuning if you want a standalone merged model and a quantized `.gguf` file for manual Hugging Face upload.

Outputs:
- merged Hugging Face model folder
- f16 GGUF
- Q4_K_S GGUF


In [ ]:
# Optional export: merge LoRA adapter into the Gemma base model.
# This produces a full Hugging Face model folder. It can be large.
RUN_LLM_MERGED_MODEL_EXPORT = True

if RUN_LLM_MERGED_MODEL_EXPORT and CFG.RUN_LLM_FINE_TUNE:
    fine_tune_run_name = llm_summary_df.iloc[0]["experiment_name"]
    adapter_dir = Path(llm_summary_df.iloc[0]["adapter_dir"])
    merged_model_dir = CFG.LLM_OUTPUT_ROOT / f"{fine_tune_run_name}_merged_hf"
    merged_model_dir.mkdir(parents=True, exist_ok=True)

    print(f"Adapter source: {adapter_dir}")
    print(f"Merged model output: {merged_model_dir}")

    cleanup_cuda()

    merge_dtype = torch.bfloat16 if IS_BF16_CAPABLE else torch.float16
    base_kwargs = dict(
        device_map="auto",
        token=CFG.HF_TOKEN,
        attn_implementation="sdpa",
    )

    try:
        base_model = AutoModelForCausalLM.from_pretrained(
            CFG.LLM_MODEL_ID,
            dtype=merge_dtype,
            **base_kwargs,
        )
    except TypeError:
        base_model = AutoModelForCausalLM.from_pretrained(
            CFG.LLM_MODEL_ID,
            torch_dtype=merge_dtype,
            **base_kwargs,
        )

    peft_model = PeftModel.from_pretrained(base_model, str(adapter_dir))
    merged_model = peft_model.merge_and_unload()
    merged_model.eval()

    merged_model.save_pretrained(
        merged_model_dir,
        safe_serialization=True,
        max_shard_size="4GB",
    )
    processor.save_pretrained(merged_model_dir)

    export_manifest = {
        "source_base_model": CFG.LLM_MODEL_ID,
        "source_adapter_dir": str(adapter_dir),
        "merged_model_dir": str(merged_model_dir),
        "export_dtype": "bfloat16" if IS_BF16_CAPABLE else "float16",
        "experiment_name": fine_tune_run_name,
    }
    with open(merged_model_dir / "haryanvi_export_manifest.json", "w", encoding="utf-8") as f:
        json.dump(export_manifest, f, ensure_ascii=False, indent=2)

    print("Merged HF model export complete.")
    print(f"Upload folder manually with:")
    print(f"huggingface-cli upload your-org/{fine_tune_run_name}-merged {merged_model_dir}")

    del merged_model
    del peft_model
    del base_model
    cleanup_cuda()
elif RUN_LLM_MERGED_MODEL_EXPORT:
    print("Skipping merged model export because CFG.RUN_LLM_FINE_TUNE is False.")


In [ ]:
# Optional export: convert merged HF model to GGUF and quantize to Q4_K_S.
# Requires enough disk for the merged model plus intermediate f16 GGUF.
RUN_LLM_GGUF_EXPORT = True

import subprocess


def run_cmd(command, cwd=None):
    print("$ " + " ".join(str(part) for part in command))
    subprocess.run(command, cwd=str(cwd) if cwd else None, check=True)


if RUN_LLM_GGUF_EXPORT and CFG.RUN_LLM_FINE_TUNE:
    fine_tune_run_name = llm_summary_df.iloc[0]["experiment_name"]
    merged_model_dir = CFG.LLM_OUTPUT_ROOT / f"{fine_tune_run_name}_merged_hf"
    gguf_output_dir = CFG.LLM_OUTPUT_ROOT / "gguf_export"
    llama_cpp_dir = CFG.LLM_OUTPUT_ROOT / "llama.cpp"

    gguf_output_dir.mkdir(parents=True, exist_ok=True)
    gguf_f16_path = gguf_output_dir / f"{fine_tune_run_name}.f16.gguf"
    gguf_q4_k_s_path = gguf_output_dir / f"{fine_tune_run_name}.Q4_K_S.gguf"

    if not merged_model_dir.exists():
        raise FileNotFoundError(
            f"Merged model folder not found: {merged_model_dir}. "
            "Run the merged-model export cell first."
        )

    if not llama_cpp_dir.exists():
        run_cmd([
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/ggerganov/llama.cpp.git",
            str(llama_cpp_dir),
        ])
    else:
        print(f"Using existing llama.cpp checkout: {llama_cpp_dir}")

    convert_requirements = llama_cpp_dir / "requirements" / "requirements-convert_hf_to_gguf.txt"
    if convert_requirements.exists():
        run_cmd([sys.executable, "-m", "pip", "install", "-r", str(convert_requirements)])

    build_dir = llama_cpp_dir / "build"
    run_cmd(["cmake", "-S", str(llama_cpp_dir), "-B", str(build_dir), "-DCMAKE_BUILD_TYPE=Release"])
    run_cmd(["cmake", "--build", str(build_dir), "--config", "Release", "-j", "2"])

    convert_script = llama_cpp_dir / "convert_hf_to_gguf.py"
    if not convert_script.exists():
        raise FileNotFoundError(f"llama.cpp converter not found: {convert_script}")

    run_cmd([
        sys.executable,
        str(convert_script),
        str(merged_model_dir),
        "--outfile",
        str(gguf_f16_path),
        "--outtype",
        "f16",
    ])

    quantize_candidates = [
        build_dir / "bin" / "llama-quantize",
        build_dir / "bin" / "quantize",
        llama_cpp_dir / "llama-quantize",
        llama_cpp_dir / "quantize",
    ]
    quantize_bin = next((path for path in quantize_candidates if path.exists()), None)
    if quantize_bin is None:
        raise FileNotFoundError(
            "Could not find llama.cpp quantize binary. Checked: "
            + ", ".join(str(path) for path in quantize_candidates)
        )

    run_cmd([
        str(quantize_bin),
        str(gguf_f16_path),
        str(gguf_q4_k_s_path),
        "Q4_K_S",
    ])

    print("GGUF export complete.")
    print(f"f16 GGUF: {gguf_f16_path}")
    print(f"Q4_K_S GGUF: {gguf_q4_k_s_path}")
    print("Manual Hugging Face upload command:")
    print(f"huggingface-cli upload your-org/{fine_tune_run_name}-gguf {gguf_q4_k_s_path}")

    try:
        from google.colab import files
        files.download(str(gguf_q4_k_s_path))
    except Exception:
        print("Not running in Colab download context; use the path above.")
elif RUN_LLM_GGUF_EXPORT:
    print("Skipping GGUF export because CFG.RUN_LLM_FINE_TUNE is False.")
